In [0]:
# %pip install phonenumbers pycountry rapidfuzz

In [0]:
import sys
sys.path.append("..")

In [0]:
from utils.silver_functions import *
from pyspark.sql import functions as F

In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_suppliers")
display(df)

In [0]:
#Profiling rapide
null_counts = df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# Profiling des doublons métier

# Pour voir les groupes de doublons en excluant supplier_id, created_at, updated_at :

business_cols = [
    "supplier_name",
    "city",
    "country",
    "country_code",
]

business_dup = (
    df.groupBy(business_cols)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(business_dup)

In [0]:
# Normalisation des données (supplier_name, city, country, contact_name+surnam )
df_clean = (
    df
    .withColumn("supplier_name", normalize_string_udf(F.col("supplier_name")))
    .withColumn("city", normalize_string_udf(F.col("city")))
    .withColumn("country", normalize_string_udf(F.col("country")))
    .withColumn("contact_name", normalize_string_udf(F.col("contact_name")))
    .withColumn("contact_surname", normalize_string_udf(F.col("contact_surname")))
)

In [0]:
# Nettoyer les dates :
df_clean = (
    df_clean
    .withColumn("created_at", normalize_date("created_at"))
    .withColumn("updated_at", normalize_date("updated_at"))
)

In [0]:
# Correction city & country par fréquence
# ==============================================================
correct_city_udf    = build_freq_corrector(df_clean, "city")
correct_country_udf = build_freq_corrector(df_clean, "country")

df_clean = (
    df_clean
    .withColumn("city",    correct_city_udf(F.col("city")))
    .withColumn("country", correct_country_udf(F.col("country")))
)

In [0]:
# Normalisation country_code, phone, email

# Country code :
# Recalculé à partir de country plutôt que faire confiance à la colonne brute.

# Phone number :
# Pour que phonenumbers marche mieux, il faut lui donner un pays par défaut cohérent avec le country_code.

email_regex = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
df_clean = (
    df_clean
    .withColumn("country_code", normalize_country_code_udf(F.col("country")))
    .withColumn("phone_number", normalize_phone_with_country_udf(F.col("phone_number"), F.col("country_code")))
    .withColumn(
        "contact_email",
        F.when(F.lower(F.trim(F.col("contact_email"))).isin("", "null", "n/a", "unknown"), None)
         .when(F.lower(F.trim(F.col("contact_email"))).rlike(email_regex), F.lower(F.trim(F.col("contact_email"))))
         .otherwise(None)
    )
)

In [0]:
# Déduplication métier


# Étape 2
# Dédupliquer les doublons métier sur :
# toutes les colonnes descriptives en excluant supplier_id, created_at, updated_at

# Étape 3
# Garder la ligne la plus récente selon :
# Updated_at
# sinon created_at

df_clean = df_clean.withColumn(
    "last_change_date",
    F.coalesce(F.col("updated_at"), F.col("created_at"))
)

w = Window.partitionBy(*business_cols).orderBy(
    F.col("last_change_date").desc_nulls_last(),
    F.col("supplier_id").asc_nulls_last()
)

df_ranked = df_clean.withColumn("rn", F.row_number().over(w))

In [0]:
# ==============================================
# MISE À JOUR DES ORDERS POUR COHÉRENCE SUPPLIER_ID
# ==============================================

# 1️ Récupérer les fournisseurs conservés (rn == 1)
df_suppliers_kept = df_ranked.filter(F.col("rn") == 1).select("supplier_id", *business_cols)

# 2️ Récupérer les doublons métier (rn > 1) et créer le mapping old_supplier_id -> new_supplier_id
df_mapping = (
    df_ranked.filter(F.col("rn") > 1).alias("dup")
    .join(
        df_suppliers_kept.alias("kept"),
        on=business_cols,
        how="inner"
    )
    .select(
        F.col("dup.supplier_id").alias("old_supplier_id"),
        F.col("kept.supplier_id").alias("new_supplier_id")
    )
)

display(df_mapping)  # pour vérifier que le mapping est correct

# 3️ Charger la table bronze_orders
orders = spark.table("workspace.final_project.bronze_orders")

# 4️ Mettre à jour les supplier_id dans orders selon le mapping
orders_silver = (
    orders.alias("o").join(
        df_mapping.alias("m"),
        orders.supplier_id == F.col("m.old_supplier_id"),
        how="left"
    )
    .withColumn(
        "supplier_id",
        F.coalesce(F.col("m.new_supplier_id"), F.col("o.supplier_id"))
    )
    .drop("old_supplier_id", "new_supplier_id")
)

# Vérification rapide
display(orders_silver.limit(10))

# 5️ Écriture de la table Silver Orders
orders_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.final_project.orders_prepared_for_silver")
print(" Orders préparées pour le notebook Silver Orders")

In [0]:
# Snapshot après nettoyage
df_after_cleaning = df_clean



# -----------------------------
# 3. DETECTION DES DOUBLONS
# -----------------------------

df_valid = df_after_cleaning.withColumn(
    "last_change_date",
    F.coalesce(F.col("updated_at"), F.col("created_at"))
)

w = Window.partitionBy(*business_cols).orderBy(
    F.col("last_change_date").desc_nulls_last(),
    F.col("supplier_id").asc_nulls_last()
)

df_ranked = df_valid.withColumn("rn", F.row_number().over(w))


# Colonnes originales sans supplier_id
cols_without_id = [c for c in df_clean.columns if c != "supplier_id"]

quarantine_duplicates = (
    df_ranked
    .filter(F.col("rn") > 1)          # seulement les doublons
    .select("supplier_id", *cols_without_id)
    .withColumn("quarantine_reason", F.lit("DUPLICATE_ON_BUSINESS_COLUMNS"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

display(quarantine_duplicates)

# Quarantaine doublons
quarantine_duplicates.write.format("delta").mode("overwrite").saveAsTable("workspace.final_project.quarantine_suppliers")

# -----------------------------
# 5. DATASET FINAL SILVER
# -----------------------------

df_clean = (
    df_ranked
    .filter(F.col("rn") == 1)
    .drop("rn", "last_change_date")
)

In [0]:
#Contrôles finaux
#Nulls
final_null_counts = df_clean.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df_clean.columns
])

display(final_null_counts)

# supplier_id unique
check_dup = (
    df_clean.groupBy("supplier_id")
    .count()
    .filter(F.col("count") > 1)
)

display(check_dup)
# Écriture Silver
df_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.final_project.silver_suppliers")

In [0]:
became_null = (
    df.alias("before")
    .join(df_clean.alias("after"), on="supplier_id", how="inner")
    .filter(
        F.col("before.contact_name").isNotNull() &
        F.col("after.contact_name").isNull()
    )
    .select(
        "supplier_id",
        F.col("before.contact_name").alias("contact_name_before"),
        F.col("after.contact_name").alias("contact_name_after")
    )
)

display(became_null)

In [0]:
from functools import reduce
from pyspark.sql import functions as F

# Colonnes à auditer
cols_to_audit = [
    "supplier_name",
    "contact_name",
    "contact_email",
    "city",
    "country",
    "country_code",
    "phone_number",
    "created_at",
    "updated_at"
]

# Base avant / après
base = df.alias("before").join(df_clean.alias("after"), on="supplier_id", how="inner")

audit_tables = []

for c in cols_to_audit:
    tmp = (
        base
        .filter(
            F.coalesce(F.col(f"before.{c}").cast("string"), F.lit("__NULL__"))
            !=
            F.coalesce(F.col(f"after.{c}").cast("string"), F.lit("__NULL__"))
        )
        .select(
            F.col("before.supplier_id").alias("supplier_id"),  # Ajouté ici
            F.lit(c).alias("column_name"),
            F.col(f"before.{c}").cast("string").alias("before_value"),
            F.col(f"after.{c}").cast("string").alias("after_value"),
            F.when(
                F.col(f"before.{c}").isNotNull() & F.col(f"after.{c}").isNull(),
                F.lit("TO_NULL")
            ).otherwise(F.lit("CHANGED")).alias("change_type")
        )
    )
    audit_tables.append(tmp)

# Unionner tous les résultats
audit_long = reduce(lambda a, b: a.unionByName(b), audit_tables)

# Affichage final avec supplier_id
display(audit_long.orderBy("column_name"))

In [0]:
df_clean.display()